# Chapter 14 — A Successful Call Is Not Finished Work

**Companion to *Applied AI*.**

This notebook accompanies Chapter 14. The chapter separates four facts that
usually share one status field, and it backs that with a pinned run: a producer
killed at a durable checkpoint, reopened by other processes, and attacked
twelve ways.

That run is preserved here.

## Question

**What facts must exist before work can be called complete?**

## What this notebook establishes

- From the preserved run: a call that **succeeded** beside a task that is
  **incomplete**, after both a clean exit and a killed producer — with the
  inspecting process appending nothing and calling no model.
- The twelve attacks, read back with their exact refusal reasons, including the
  two that carry most of the argument: an **edited artifact that passes its own
  check**, and a **bare completion event appended by hand**.
- An acceptance validator implemented small enough to read, with the same
  refusals, so you can break it yourself.

## What this notebook does **not** establish

- The model output in the preserved run was **scripted**. The recorded path is
  real; nothing here measures repair quality.
- **Roles are structure, not security.** A label is a string, the ledger is
  unsigned, and anyone who can write the SQLite file could append a matching
  pair. The chapter says the ledger is the trust boundary.
- Crash coverage is narrow: the kill happened at a durable checkpoint, and the
  gap between the two appends was exercised by an injected exception.

## Setup

In [1]:
import hashlib
import json
import os
from pathlib import Path

def find_evidence_dir(marker="task-completion"):
    env = os.environ.get("APPLIED_AI_EVIDENCE")
    if env and Path(env).expanduser().is_dir():
        return Path(env).expanduser()
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for cand in (base / "evidence",
                     base / "experiments" / "applied-ai" / "evidence"):
            if (cand / marker).is_dir():
                return cand
    raise FileNotFoundError("Set APPLIED_AI_EVIDENCE to the evidence directory.")

EVIDENCE_DIR = find_evidence_dir()
report = json.loads(
    (EVIDENCE_DIR / "task-completion" / "offline-fe0797d" / "report.json").read_text(encoding="utf-8"))

print("chains    :", list(report["chains"]))
print("negatives :", len(report["negatives"]))
print("all passed:", report["all_passed"])

chains    : ['clean-exit', 'abrupt-termination']
negatives : 12
all passed: True


## 1. Four facts that are not the same fact

```text
generation succeeded  !=  artifact checked  !=  artifact accepted  !=  task completed
```

The output belongs to the call. **"Done" belongs to the task.**

In [2]:
OWNERS = [
    ("Call",       "generated artifact",  "what the model produced"),
    ("Check",      "evidence",            "what a deterministic procedure observed"),
    ("Acceptance", "a decision",          "cites call, artifact and check; made by an eligible role"),
    ("Completion", "valid only when caused by that acceptance", "derived, never set"),
]
for who, what, note in OWNERS:
    print(f"  {who:<12}{what:<44}{note}")

print()
print("A model is very good at producing the first line. It has no access to")
print("the other three: it cannot know whether its bytes were checked, which")
print("bytes the checker saw, whether anyone with standing agreed, or whether")
print("that agreement was recorded.")

  Call        generated artifact                          what the model produced
  Check       evidence                                    what a deterministic procedure observed
  Acceptance  a decision                                  cites call, artifact and check; made by an eligible role
  Completion  valid only when caused by that acceptance   derived, never set

A model is very good at producing the first line. It has no access to
the other three: it cannot know whether its bytes were checked, which
bytes the checker saw, whether anyone with standing agreed, or whether
that agreement was recorded.


## 2. The pinned run: a correct answer, an unfinished task

Two producers, both interrupted differently. Both reopened by a **separate
process** that was given only the files.

In [3]:
for mode, chain in report["chains"].items():
    a = chain["assertions"]
    print(f"=== {mode} ===")
    print(f"  termination as labelled       : {a['termination_as_labelled']}")
    print(f"  call succeeded                : {a['call_succeeded']}")
    print(f"  generation complete           : {a['generation_complete']}")
    print(f"  TASK INCOMPLETE after reopen  : {a['task_incomplete_after_reopen']}")
    print(f"  inspecting appended nothing   : {a['inspect_appended_nothing']}")
    print(f"  cognition NOT re-run          : {a['cognition_not_rerun']}")
    print()

for mode, chain in report["chains"].items():
    a = chain["assertions"]
    assert a["call_succeeded"] and a["task_incomplete_after_reopen"]
    assert a["inspect_appended_nothing"] and not a["cognition_not_rerun"] is None

print("assertion held: in both modes the call succeeded and the task did not")
print("complete, and reading the process changed nothing.")

=== clean-exit ===
  termination as labelled       : True
  call succeeded                : True
  generation complete           : True
  TASK INCOMPLETE after reopen  : True
  inspecting appended nothing   : True
  cognition NOT re-run          : True

=== abrupt-termination ===
  termination as labelled       : True
  call succeeded                : True
  generation complete           : True
  TASK INCOMPLETE after reopen  : True
  inspecting appended nothing   : True
  cognition NOT re-run          : True

assertion held: in both modes the call succeeded and the task did not
complete, and reading the process changed nothing.


## Observation

Nothing went wrong in either run. The model answered correctly, the dialect was
handled, generation finished normally, and the attempt was interpreted and
preserved.

**The task is still not complete**, because no acceptance was recorded.

That is the line Chapter 11 ended on — `task_status: not automatically
completed` — made executable.

In [4]:
clean = report["chains"]["clean-exit"]["cognition_event_counts_before_after"]
print("cognition events, before and after the inspecting process ran:")
for kind, (before, after) in clean.items():
    flag = "unchanged" if before == after else "CHANGED"
    print(f"  {kind:<22} {before} -> {after}   {flag}")
print()
print("Restarting invented no acceptance, and it did not re-run the cognition")
print("'to be safe'. A model call is an increment, not a restartable store.")

cognition events, before and after the inspecting process ran:
  call.manifest          1 -> 1   unchanged
  attempt.started        1 -> 1   unchanged
  attempt.observed       1 -> 1   unchanged
  attempt.completed      1 -> 1   unchanged

Restarting invented no acceptance, and it did not re-run the cognition
'to be safe'. A model call is an increment, not a restartable store.


## 3. What acceptance had to cite

Once a check has run, acceptance names an exact chain. Every field is an
identity the runtime can look up.

In [5]:
CHAIN = ["task", "call", "final attempt", "interpretation",
         "artifact sha256", "check(s)", "acceptor role"]
print(" -> ".join(CHAIN))
print()
CONDITIONS = [
    "authority grants ACCEPT, a capability separate from producing work",
    "the acceptor label differs from the producing actor's label",
    "the acceptance cites the hash of the task's DECLARED criteria",
    "the source call belongs to this task and its status is succeeded",
    "the cited attempt is the call's final attempt",
    "the cited interpretation is the one the call's status rests on",
    "the artifact sha256 equals the preserved output, and the bytes are intact",
    "at least one check, recorded for this task, targeted THOSE bytes, PASSed",
]
for i, c in enumerate(CONDITIONS, 1):
    print(f"  {i}. {c}")
print()
print("Nothing in the acceptance says the paragraph is good. That judgment")
print("lives in the check it cites.")

task -> call -> final attempt -> interpretation -> artifact sha256 -> check(s) -> acceptor role

  1. authority grants ACCEPT, a capability separate from producing work
  2. the acceptor label differs from the producing actor's label
  3. the acceptance cites the hash of the task's DECLARED criteria
  4. the source call belongs to this task and its status is succeeded
  5. the cited attempt is the call's final attempt
  6. the cited interpretation is the one the call's status rests on
  7. the artifact sha256 equals the preserved output, and the bytes are intact
  8. at least one check, recorded for this task, targeted THOSE bytes, PASSed

Nothing in the acceptance says the paragraph is good. That judgment
lives in the check it cites.


## 4. Twelve ways to cheat

Each ran in its own ledger. Apart from the last, none produced an acceptance
and none projects as completed.

In [6]:
LABELS = {
    "no-acceptance": "Succeeded call and passing check, nobody accepts",
    "missing-check": "Acceptance cites no check",
    "failed-check": 'Output kept "73%"; the real check returned FAIL',
    "error-check": "The check command could not run",
    "check-for-other-task": "Passing check on the right bytes, wrong task",
    "call-from-other-task": "Acceptance cites another task's call and output",
    "changed-artifact": "An EDITED paragraph that passes all three criteria",
    "unauthorized": "Acceptor holds every capability except ACCEPT",
    "self-acceptance": "The producing actor's label tries to accept",
    "truncated-generation": "finish_reason=length, text still passes the check",
    "completion-without-acceptance": "A bare task.completed appended by hand",
    "interrupted-between-appends": "Failure between acceptance and completion",
}

print(f"{'case':<32}{'refusal reason(s) recorded':<58}{'complete?'}")
print("-" * 104)
for n in report["negatives"]:
    reasons = n["reasons"]
    shown = ", ".join(reasons) if reasons else "(nothing to reject)"
    if len(shown) > 55:
        shown = shown[:52] + "..."
    done = "only after repeat" if n["case"] == "interrupted-between-appends" else "No"
    print(f"{n['case']:<32}{shown:<58}{done}")

assert all(n["passed"] for n in report["negatives"])
print()
print("assertion held: all twelve behaved as the protocol required")

case                            refusal reason(s) recorded                                complete?
--------------------------------------------------------------------------------------------------------
no-acceptance                   (nothing to reject)                                       No
missing-check                   missing_check                                             No
failed-check                    check_not_passed:check-c10f5bdc-b610-4e88-b8a1-9548b...   No
error-check                     check_not_passed:check-1e1f6bea-0622-4fd8-928b-5cce2...   No
check-for-other-task            check_wrong_task:check-d157e7eb-02dd-4118-8f03-9a7c9...   No
call-from-other-task            source_call_wrong_task                                    No
changed-artifact                artifact_not_source_output                                No
unauthorized                    unauthorized                                              No
self-acceptance                 self_acceptance    

## The three that carry the argument

In [7]:
for case in ("no-acceptance", "changed-artifact", "completion-without-acceptance"):
    n = next(x for x in report["negatives"] if x["case"] == case)
    print(f"--- {case} ---")
    print(f"    {LABELS[case]}")
    print(f"    recorded: {n['reasons']}")
    print()

print("A SUCCEEDED CALL IS INCOMPLETE. No failure is needed to show this:")
print("the call succeeded and the work was not done.")
print()
print("THE EDITED ARTIFACT passed its check and was still refused. The edit was")
print("small - 'is intended to make' became 'should make' - and the criteria")
print("check passed on the edited bytes. That check was perfectly valid. It was")
print("also about a DIFFERENT ENTITY from the one the call generated.")
print()
print("A BARE COMPLETION completes nothing. The projection asked what acceptance")
print("caused it. There was no answer, so the task stayed incomplete.")

--- no-acceptance ---
    Succeeded call and passing check, nobody accepts
    recorded: None

--- changed-artifact ---
    An EDITED paragraph that passes all three criteria
    recorded: ['artifact_not_source_output']

--- completion-without-acceptance ---
    A bare task.completed appended by hand
    recorded: None

A SUCCEEDED CALL IS INCOMPLETE. No failure is needed to show this:
the call succeeded and the work was not done.

THE EDITED ARTIFACT passed its check and was still refused. The edit was
small - 'is intended to make' became 'should make' - and the criteria
check passed on the edited bytes. That check was perfectly valid. It was
also about a DIFFERENT ENTITY from the one the call generated.

A BARE COMPLETION completes nothing. The projection asked what acceptance
caused it. There was no answer, so the task stayed incomplete.


## 5. Build it yourself

Small enough to read, with the same refusals. The point is that you can break
it in the next cell.

In [8]:
def sha256(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

class Ledger(list):
    def append_event(self, kind, **payload):
        self.append({"seq": len(self) + 1, "kind": kind, **payload})
        return self[-1]

def accept_task(ledger, *, task_id, criteria_sha, artifact_sha, artifact_bytes,
                source_call_id, check_ids, actor_id, capabilities,
                producing_actor, calls, checks, criteria_by_task):
    reasons = []
    if "accept" not in capabilities:
        reasons.append("unauthorized")
    if actor_id == producing_actor:
        reasons.append("self_acceptance")
    if criteria_by_task.get(task_id) != criteria_sha:
        reasons.append("criteria_mismatch")

    call = calls.get(source_call_id)
    if call is None or call["task_id"] != task_id:
        reasons.append("source_call_wrong_task")
    elif call["status"] != "succeeded":
        reasons.append("source_call_not_succeeded")
    elif call["generation"] != "complete":
        reasons.append("generation_not_complete")
    elif call["output_sha"] != artifact_sha:
        reasons.append("artifact_not_source_output")

    if sha256(artifact_bytes) != artifact_sha:
        reasons.append("artifact_bytes_do_not_match")

    if not check_ids:
        reasons.append("missing_check")
    for cid in check_ids:
        c = checks.get(cid)
        if c is None:
            reasons.append(f"check_missing:{cid}")
        elif c["task_id"] != task_id:
            reasons.append(f"check_wrong_task:{cid}")
        elif c["target_sha"] != artifact_sha:
            reasons.append(f"check_wrong_target:{cid}")
        elif c["verdict"] != "PASS":
            reasons.append(f"check_not_passed:{cid}:{c['verdict']}")

    if reasons:
        ledger.append_event("task.acceptance_rejected", task_id=task_id,
                            reasons=reasons)
        return False, reasons

    acc = ledger.append_event("task.accepted", task_id=task_id, actor_id=actor_id,
                              artifact_sha=artifact_sha, check_ids=list(check_ids))
    ledger.append_event("task.completed", task_id=task_id,
                        causation_id=acc["seq"])
    return True, []

def is_complete(ledger, task_id):
    """Derived, never set. A completion with no causing acceptance counts for nothing."""
    accepted = {e["seq"] for e in ledger
                if e["kind"] == "task.accepted" and e["task_id"] == task_id}
    return any(e["kind"] == "task.completed" and e["task_id"] == task_id
               and e.get("causation_id") in accepted for e in ledger)

In [9]:
GOOD = b"The new cache is intended to make page loads faster [S1].\nIt stores fragments.\n"
EDIT = b"The new cache should make page loads faster [S1].\nIt stores fragments.\n"

calls = {"call-1": {"task_id": "t1", "status": "succeeded",
                    "generation": "complete", "output_sha": sha256(GOOD)}}
criteria = {"t1": "criteria-v1"}

def fresh_checks(target_sha, verdict="PASS", task_id="t1"):
    return {"chk-1": {"task_id": task_id, "target_sha": target_sha, "verdict": verdict}}

led = Ledger()
ok, why = accept_task(
    led, task_id="t1", criteria_sha="criteria-v1", artifact_sha=sha256(GOOD),
    artifact_bytes=GOOD, source_call_id="call-1", check_ids=["chk-1"],
    actor_id="reviewer", capabilities={"accept"}, producing_actor="repairer",
    calls=calls, checks=fresh_checks(sha256(GOOD)), criteria_by_task=criteria)

print("happy path       :", ok, why)
print("events appended  :", [e["kind"] for e in led])
print("task complete    :", is_complete(led, "t1"))
assert ok and is_complete(led, "t1")

happy path       : True []
events appended  : ['task.accepted', 'task.completed']
task complete    : True


## 6. Break it

The edited artifact, with a check that genuinely passes on the edited bytes.

In [10]:
led2 = Ledger()
ok2, why2 = accept_task(
    led2, task_id="t1", criteria_sha="criteria-v1", artifact_sha=sha256(EDIT),
    artifact_bytes=EDIT, source_call_id="call-1", check_ids=["chk-1"],
    actor_id="reviewer", capabilities={"accept"}, producing_actor="repairer",
    calls=calls, checks=fresh_checks(sha256(EDIT)), criteria_by_task=criteria)

print("edited artifact  :", ok2, why2)
print("task complete    :", is_complete(led2, "t1"))
print()
print("The check PASSED on those exact bytes. It was still refused, because")
print("the accepted entity is not the one the call generated.")

edited artifact  : False ['artifact_not_source_output']
task complete    : False

The check PASSED on those exact bytes. It was still refused, because
the accepted entity is not the one the call generated.


In [11]:
cases = [
    ("producer accepts its own output", dict(actor_id="repairer")),
    ("acceptor lacks ACCEPT",           dict(capabilities={"read", "write"})),
    ("no check cited",                  dict(check_ids=[])),
]
for label, override in cases:
    kwargs = dict(task_id="t1", criteria_sha="criteria-v1",
                  artifact_sha=sha256(GOOD), artifact_bytes=GOOD,
                  source_call_id="call-1", check_ids=["chk-1"],
                  actor_id="reviewer", capabilities={"accept"},
                  producing_actor="repairer", calls=calls,
                  checks=fresh_checks(sha256(GOOD)), criteria_by_task=criteria)
    kwargs.update(override)
    l = Ledger()
    ok3, why3 = accept_task(l, **kwargs)
    print(f"{label:<34} refused: {why3}")

producer accepts its own output    refused: ['self_acceptance']
acceptor lacks ACCEPT              refused: ['unauthorized']
no check cited                     refused: ['missing_check']


## A bare completion completes nothing

In [12]:
forged = Ledger()
forged.append_event("task.completed", task_id="t1")          # no causation
print("events:", [e["kind"] for e in forged])
print("task complete:", is_complete(forged, "t1"))
assert not is_complete(forged, "t1")
print()
print("assertion held: anyone can append an event. Completion is DERIVED from a")
print("completion caused by a recorded acceptance, so an uncaused one is a")
print("claim, not a fact.")

events: ['task.completed']
task complete: False

assertion held: anyone can append an event. Completion is DERIVED from a
completion caused by a recorded acceptance, so an uncaused one is a
claim, not a fact.


## Interpretation

The chapter's end-to-end argument, from Saltzer, Reed and Clark: transport
success, a parsed body, a normal finish reason and a well-shaped answer are
**per-hop checksums**. They are worth having. None of them is the end-to-end
check.

The end is the task. The check has to examine **the artifact that will be
used**, against **the task's own criteria**, and the commit decision has to
follow from that.

Four rules survive into the rest of the book:

1. **Check at the end**, on the bytes you are accepting.
2. **Derive "done"; do not set it.** A completion with no cause is a claim.
3. **Keep producer and acceptor apart** — as structure, not as security.
4. **An identical repeat must change nothing**, and the gap between accepting
   and completing must be recoverable, because they are two separate writes.

## Try it yourself

1. **Make the criteria mutable.** Accept, then change `criteria` and re-derive
   completion. Nothing breaks — which is why acceptance binds the criteria
   *hash*, not the criteria.
2. **Add idempotency.** Send the identical acceptance twice and make the second
   append nothing. Then send a *different* acceptance for the same task and
   refuse it. That is Helland's at-least-once delivery, and Chapter 22's
   subject.
3. **Forge properly.** Append both `task.accepted` and a `task.completed` that
   cites it. `is_complete` now returns True. You have just demonstrated the
   chapter's own stated limit: the ledger is the trust boundary.
4. **Weaken the check.** Change the criteria check to test only that `[S1]`
   appears once. Which of the twelve attacks now succeed? Chapter 21 is about
   exactly that gap.